# Federated Learning with Tiny Batches

    "metadata": {"language": "markdown"},

The default experiment uses synthetic data, so you can learn the mechanics without loading the large CSV files. Later cells show how to switch to a very small Parquet sample from `data/splits/train.parquet`. Tiny batches are useful for learning and debugging; they are not meaningful evidence of model quality.

## Learning goals

    "metadata": {"language": "markdown"},

1. Why each client starts a round from the same global model.
2. How local updates differ when client data is non-IID.
3. How FedAvg combines client models using the number of examples as weights.
4. Why macro-F1 and per-class recall matter more than accuracy for this imbalanced security dataset.

    "metadata": {"language": "python"},

In [ ]:
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, f1_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 7
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES = 8
NUM_CLIENTS = 3
BATCH_SIZE = 8
LOCAL_EPOCHS = 1
print(f'Device: {DEVICE}')

## 1. Start with a tiny synthetic dataset

Each class is represented by a small cluster in feature space. The synthetic task is deliberately simple: it lets you inspect the federated mechanics before adding dataset-specific preprocessing.

In [ ]:
def make_synthetic_data(samples_per_class=30, features=12):
    generator = torch.Generator().manual_seed(SEED)
    centers = torch.randn(NUM_CLASSES, features, generator=generator) * 2.0
    rows, labels = [], []
    for class_id in range(NUM_CLASSES):
        noise = torch.randn(samples_per_class, features, generator=generator) * 0.7
        rows.append(centers[class_id] + noise)
        labels.append(torch.full((samples_per_class,), class_id, dtype=torch.long))
    return torch.cat(rows), torch.cat(labels)

features, labels = make_synthetic_data()
print('features:', tuple(features.shape))
print('labels:', tuple(labels.shape))
print('class counts:', torch.bincount(labels, minlength=NUM_CLASSES).tolist())

## 2. Create non-IID virtual clients

Non-IID means clients do not see identical class mixtures. Here, client 0 gets mostly classes 0-2, client 1 mostly 3-5, and client 2 mostly 6-7, while a small number of examples are shared across groups. This is a teaching approximation of different attack-category mixes, not a claim about actual devices.

In [ ]:
def make_non_iid_clients(x, y):
    client_classes = [[0, 1, 2, 3], [2, 3, 4, 5], [4, 5, 6, 7]]
    clients = []
    for class_group in client_classes:
        mask = torch.zeros_like(y, dtype=torch.bool)
        for class_id in class_group:
            mask |= y == class_id
        client_x, client_y = x[mask], y[mask]
        order = torch.randperm(len(client_y), generator=torch.Generator().manual_seed(SEED))
        clients.append((client_x[order], client_y[order]))
    return clients

clients = make_non_iid_clients(features, labels)
for client_id, (_, client_y) in enumerate(clients):
    print(f'client {client_id}: {len(client_y)} rows, counts={torch.bincount(client_y, minlength=NUM_CLASSES).tolist()}')

## 3. Define a deliberately small model

The model is an MLP with one hidden layer. Every client must use the same architecture because their parameters need to be compatible when the server averages them.

In [ ]:
class SmallMLP(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_features, 24),
            nn.ReLU(),
            nn.Linear(24, NUM_CLASSES),
        )

    def forward(self, x):
        return self.network(x)

def make_model(input_features):
    return SmallMLP(input_features).to(DEVICE)

## 4. Local training and FedAvg

A round has three stages:

- The server broadcasts the current global weights.
- Each client trains a copy for `LOCAL_EPOCHS` using only its own data.
- The server computes a weighted average of corresponding tensors.

The weighted average is `sum(client_weights * client_row_count) / total_row_count`. This gives larger clients proportionally more influence.

In [ ]:
def train_one_client(global_model, client_data, batch_size=BATCH_SIZE, epochs=LOCAL_EPOCHS):
    local_model = deepcopy(global_model).to(DEVICE)
    local_model.train()
    x, y = client_data
    loader = DataLoader(TensorDataset(x, y), batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.SGD(local_model.parameters(), lr=0.05)
    loss_fn = nn.CrossEntropyLoss()
    total_loss = 0.0
    for _ in range(epochs):
        for batch_x, batch_y in loader:
            optimizer.zero_grad()
            loss = loss_fn(local_model(batch_x.to(DEVICE)), batch_y.to(DEVICE))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
    return local_model.state_dict(), len(y), total_loss

def fed_avg(client_states, client_sizes):
    total_size = sum(client_sizes)
    averaged = {}
    for name in client_states[0]:
        averaged[name] = sum(
            state[name] * (size / total_size)
            for state, size in zip(client_states, client_sizes)
        )
    return averaged

In [ ]:
def evaluate(model, data):
    model.eval()
    x, y = data
    with torch.no_grad():
        predictions = model(x.to(DEVICE)).argmax(dim=1).cpu()
    macro_f1 = f1_score(y, predictions, average='macro', zero_division=0)
    report = classification_report(
        y, predictions, labels=list(range(NUM_CLASSES)),
        target_names=[f'class_{i}' for i in range(NUM_CLASSES)],
        zero_division=0, output_dict=True
    )
    recalls = {name: values['recall'] for name, values in report.items() if name.startswith('class_')}
    return macro_f1, recalls

global_model = make_model(features.shape[1])
initial_f1, _ = evaluate(global_model, (features, labels))
client_states, client_sizes = [], []
for client_data in clients:
    state, size, loss = train_one_client(global_model, client_data)
    client_states.append(state)
    client_sizes.append(size)
    print(f'local rows={size}, accumulated local loss={loss:.3f}')
global_model.load_state_dict(fed_avg(client_states, client_sizes))
round_f1, round_recalls = evaluate(global_model, (features, labels))
print(f'macro-F1 before round: {initial_f1:.3f}')
print(f'macro-F1 after round:  {round_f1:.3f}')
print('per-class recall:', round_recalls)

## 5. Repeat multiple federated rounds

This loop makes the round structure visible. In a real experiment you would keep a separate validation split and record metrics after every round. Do not interpret this synthetic result as a project benchmark.

In [ ]:
def run_federated_experiment(client_data, input_features, evaluation_data, rounds=5):
    model = make_model(input_features)
    history = []
    for round_id in range(1, rounds + 1):
        states, sizes = [], []
        for data in client_data:
            state, size, _ = train_one_client(model, data)
            states.append(state)
            sizes.append(size)
        model.load_state_dict(fed_avg(states, sizes))
        score, _ = evaluate(model, evaluation_data)
        history.append({'round': round_id, 'macro_f1': score})
    return model, pd.DataFrame(history)

trained_model, history = run_federated_experiment(clients, features.shape[1], (features, labels), rounds=5)
history

## 6. Optional: use a tiny real CICIoT2023 sample

Run this section only after the project sampling pipeline has created `data/splits/train.parquet`. It reads a small slice with Polars, selects numeric features, maps the existing 8-class `class` column to integer IDs, and standardizes the features. It deliberately does not read the full raw CSV into memory.

The project keeps the label mapping in `src/data/label_map.py`; this notebook does not duplicate the 34 raw-label mapping. The sampled Parquet is expected to already contain the mapped `class` column created by `sample_dataset.py`.

In [ ]:
# Optional real-data setup. Leave this cell unrun while learning with synthetic data.
import polars as pl
from src.data.label_map import CLASSES

parquet_path = Path('../data/splits/train.parquet')
real_sample = pl.read_parquet(parquet_path).head(240)
feature_names = [
    name for name, dtype in zip(real_sample.columns, real_sample.dtypes)
    if name not in {'label', 'class', 'source_file'} and dtype.is_numeric()
]
real_x = torch.tensor(real_sample.select(feature_names).to_numpy(), dtype=torch.float32)
class_to_id = {name: index for index, name in enumerate(CLASSES)}
real_y = torch.tensor([class_to_id[name] for name in real_sample['class'].to_list()], dtype=torch.long)
real_x = (real_x - real_x.mean(dim=0)) / real_x.std(dim=0).clamp_min(1e-6)
print(real_x.shape, real_y.shape, feature_names)

In [ ]:
# Optional real-data experiment. This reuses the same functions as the synthetic demo.
real_clients = make_non_iid_clients(real_x, real_y)
real_model, real_history = run_federated_experiment(real_clients, real_x.shape[1], (real_x, real_y), rounds=3)
real_history

## 7. Experiments to try

Change one setting at a time and write down what changed:

- Set `BATCH_SIZE` to 1, 4, or 16. Smaller batches make noisier local updates.
- Increase `LOCAL_EPOCHS` from 1 to 3. More local work can improve communication efficiency but may worsen client drift.
- Replace `fed_avg` with an unweighted average and compare results.
- Make the client class groups completely disjoint, then compare with overlapping groups.
- Train a centralized model on the concatenated client data and compare macro-F1, not just accuracy.
- Keep a held-out validation slice and report per-class recall after every round.
- Compare 3, 5, and 10 clients while keeping the total number of examples fixed.

For the actual project, log the seed, sampled rows, client partition rule, number of rounds, local epochs, batch size, optimizer, macro-F1, and per-class recall for every run.

## Key interpretation

A federated round is not magic distributed training: it is repeated local optimization plus parameter aggregation. The interesting project question is whether a lightweight model trained this way gives an acceptable macro-F1 and rare-class recall compared with the centralized baselines, under the same data and evaluation split.